# 025 — Training: deterministic architectures x combined_loss alpha sweep

A variant of `020_training.ipynb`. Where `020` trains each deterministic
architecture once, at the project-default loss weight
(`settings.LOSS_ALPHA` = 0.16), this notebook trains a **chosen subset** of
them (comment entries out in section 3 to filter) across a **grid of alpha
values** for the shared `combined_loss`:

    combined_loss = alpha * Charbonnier  +  (1 - alpha) * (1 - MS-SSIM)

The default grid is `{0.16, 0.50, 0.84}` — the three values first used for
`unet_residual` (`fixing.md`): MS-SSIM-heavy (the project default), balanced,
and pixel-fidelity-heavy.

**Where the checkpoints go.** Every run lands in
`models/alpha_sweep/alpha_<a>/<arch>/best_model.keras`, one directory per
alpha — `020`'s canonical `models/deterministic/<arch>/` checkpoints are
never touched. The `alpha = settings.LOSS_ALPHA` column is retrained here
too (into `alpha_sweep/`), as the sweep's anchor point and the only column
whose `val_loss` is directly comparable to `020`'s runs (same loss
function; across alphas the loss is a different function and its value is
not a ranking, same caveat as `024` §4).

**Resumable.** The `(arch, alpha)` loop skips any run whose checkpoint
already exists, so a long sweep can be run in several sessions.

| notebook | trains |
|---|---|
| `020_training.ipynb` | each deterministic architecture once, `alpha = settings.LOSS_ALPHA` -> `models/deterministic/` |
| **025** (this one) | a chosen subset x an alpha grid -> `models/alpha_sweep/alpha_<a>/` |

Evaluate the resulting checkpoints with
`035_evaluation_alpha_sweep.ipynb` (fidelity / detection AUROC / coherence,
per architecture, against the `020` default and the `mean(R, G, B)` floor).


Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports, a fixed global seed (`set_global_seed` also runs inside every subprocess, so the runs differ by architecture and alpha and by nothing else the pipeline controls), and a GPU sanity check.

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset — artwork-and-mockups split

Same split as `020` §1. Each `scripts/train_single.py` subprocess rebuilds
this split itself from `settings` (fixed seed), so nothing has to cross the
process boundary; this cell only prints the split sizes as a sanity check.

In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

print(f"Train: {len(train_pairs)} patches")
print(f"Val:   {len(val_pairs)} patches")

## 2. The alpha grid

`combined_loss(alpha)` (`scripts/losses.py`) weights a Charbonnier fidelity
term against a `(1 - MS-SSIM)` structure term. The fidelity term is
Charbonnier, i.e. L1 smoothed at `eps = settings.CHARBONNIER_EPS`, so the
ratios below carry the meaning they would with plain L1.

| label | alpha | (1 - alpha) | favours |
|---|---|---|---|
| `ssim_heavy` | 0.16 | 0.84 | structure — the Zhao et al. (2016) `Mix` ratio, `settings.LOSS_ALPHA` |
| `balanced` | 0.50 | 0.50 | neither |
| `l1_heavy` | 0.84 | 0.16 | pixel fidelity |

`--loss-alpha` is passed on the command line (not through ambient config)
so each run's alpha is recorded in the command that launched it — it does
not survive into the checkpoint.

In [ ]:
ALPHAS = [0.16, 0.50, 0.84]  # combined_loss Charbonnier weight; 0.16 == settings.LOSS_ALPHA

assert settings.LOSS_ALPHA in ALPHAS, (
    f"settings.LOSS_ALPHA ({settings.LOSS_ALPHA}) is not in the grid — add it "
    "so the sweep has an anchor comparable to 020's runs."
)
for a in ALPHAS:
    print(f"alpha={a:.2f}  charbonnier={a:.2f}  ms_ssim={1 - a:.2f}")

## 3. Train — one subprocess per (architecture, alpha)

`ARCHS` maps each architecture to its builder kwargs (mirroring `020` /
`023`). **Comment out any entry to skip that architecture.** The default
set is the four `020` deterministic architectures; the `unet_v2` /
`unet_restormer` / dilated variants are listed commented, with the kwargs
`023` trains them with.

Each `(arch, alpha)` pair trains in its own **subprocess**
(`scripts/train_single.py`), same reasoning as `020` §3 (`fixing.md` §7:
training several Keras models back-to-back in one process leaves GPU-side
state behind that `clear_session()` does not fully release on this
hardware). A pair whose checkpoint already exists is skipped, so the sweep
is resumable.

Set `EPOCHS = 2` for a quick smoke test before committing to the full grid
(`len(ARCHS) * len(ALPHAS)` full training runs — this is a long notebook).

In [ ]:
import json
import subprocess

# Comment out any architecture to skip it. Values are builder kwargs.
ARCHS = {
    "unet": {},
    "resunet": {},
    "attention_unet": {},
    "unet_residual": {},
    # "unet_v2": dict(use_strided_conv=True, use_upsample_conv=True, dropout_rate=0.2),
    # "unet_restormer": dict(num_heads=8, ffn_expansion_factor=2),
    # "unet_dilated": {},
    # "unet_v2_dilated": dict(use_strided_conv=True, use_upsample_conv=True, dropout_rate=0.2),
}

EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
SWEEP_DIR = settings.MODELS_DIR / "alpha_sweep"
SWEEP_LOG_DIR = settings.LOGS_DIR / "alpha_sweep"


def run_dirs(alpha: float) -> tuple[Path, Path]:
    tag = f"alpha_{alpha:.2f}"
    return SWEEP_DIR / tag, SWEEP_LOG_DIR / tag


histories: dict = {}

for alpha in ALPHAS:
    model_dir, log_dir = run_dirs(alpha)
    for arch, kwargs in ARCHS.items():
        ckpt = model_dir / arch / "best_model.keras"
        hist_path = model_dir / arch / "history.json"
        if ckpt.exists():
            print(f"[skip] {arch} alpha={alpha:.2f} -- exists ({ckpt})")
            histories[(arch, alpha)] = json.loads(hist_path.read_text())
            continue

        cmd = [
            sys.executable, "-m", "scripts.train_single",
            "--arch", arch,
            "--epochs", str(EPOCHS),
            "--model-dir", str(model_dir),
            "--log-dir", str(log_dir),
            "--kwargs", json.dumps(kwargs),
            "--loss-alpha", str(alpha),
        ]
        subprocess.run(cmd, cwd=project_root, check=True)

        histories[(arch, alpha)] = json.loads(hist_path.read_text())
        best = min(histories[(arch, alpha)]["val_loss"])
        print(f"\nBest val_loss ({arch}, alpha={alpha:.2f}): {best:.4f}")

## 4. Training curves

One figure per `(architecture, alpha)`. A run that diverges or plateaus in
the first few epochs is a training failure, not an alpha result — rerun it
(delete its checkpoint first) before reading anything into its metrics. The
Round 3 dilated variants (`fixing.md` #7) are the cautionary precedent.

In [ ]:
for (arch, alpha), history in histories.items():
    plot_training_curves(history, title=f"{arch} -- alpha={alpha:.2f}")
    plt.show()

## 5. Summary

`val_mae` is reported alongside `val_loss` because, unlike the loss, it does
not carry the alpha weighting and so is comparable across the grid — a
first, fidelity-only read on which alpha trained best. Expect the high-alpha
runs to win on MAE almost by construction; that is not the same as winning
on the deliverable. Rank these checkpoints in `035_evaluation_alpha_sweep.ipynb` (fidelity,
detection AUROC, coherence, per architecture) before drawing any conclusion.

In [ ]:
print(f"{'arch':<18}{'alpha':<8}{'ckpt':<9}{'epochs':<8}{'val_loss':<11}{'val_mae':<11}")
print("-" * 71)
for alpha in ALPHAS:
    model_dir, _ = run_dirs(alpha)
    for arch in ARCHS:
        h = histories.get((arch, alpha))
        if h is None:
            print(f"{arch:<18}{alpha:<8.2f}{'MISSING':<9}")
            continue
        print(f"{arch:<18}{alpha:<8.2f}{'found':<9}{len(h['val_loss']):<8}"
              f"{min(h['val_loss']):<11.4f}{min(h['val_mae']):<11.4f}")

print(f"\nCheckpoints under models/alpha_sweep/alpha_<a>/<arch>/ "
      f"(020's models/deterministic/ untouched).")
print(f"Only alpha == settings.LOSS_ALPHA ({settings.LOSS_ALPHA}) is "
      f"loss-comparable to 020's runs.")